# Sweep — Temporal context window

Proxy fold 0, lazy context loader. Context sizes 1/3/5 x 5 epochs. Heaviest sweep notebook.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.5 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Unzipped TTA embeddings to /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
import birdclef.paths as paths

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()
PROXY_FOLD = 0
EPOCHS = 5


In [4]:
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import BirdMoE, FocalLoss, competition_macro_roc_auc, seed_everything
from birdclef.context_dataset import LazyContextBirdDataset
import birdclef.paths as paths
from birdclef.sweep_plots import plot_context_sweep

SAVE_DIR = paths.SWEEPS_DIR / "context"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

WINNING_GAMMA = 1.0
WINNING_WD = 1e-4
CONTEXT_SIZES = [1, 3, 5]
train_df = df_TTA[df_TTA["fold"] != PROXY_FOLD].reset_index(drop=True)
valid_df = df_TTA[df_TTA["fold"] == PROXY_FOLD].reset_index(drop=True)

sweep_results = {}
sweep_history = {}

for ctx in CONTEXT_SIZES:
    print(f"Testing context_size={ctx}")
    train_loader = DataLoader(
        LazyContextBirdDataset(train_df, context_size=ctx),
        batch_size=64, shuffle=True, num_workers=2, pin_memory=True,
    )
    valid_loader = DataLoader(
        LazyContextBirdDataset(valid_df, context_size=ctx),
        batch_size=64, shuffle=False, num_workers=2, pin_memory=True,
    )
    input_dim = 1536 * ctx
    seed_everything(42)
    model = BirdMoE(input_dim=input_dim, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=WINNING_GAMMA)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=WINNING_WD)
    best_auc = 0.0
    val_aucs = []

    for epoch in range(EPOCHS):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if torch.isnan(inputs).any():
                continue
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        val_aucs.append(current_auc)
        best_auc = max(best_auc, current_auc)
        print(f"  Epoch {epoch + 1}/{EPOCHS} | Val AUC: {current_auc:.4f}")

    sweep_results[ctx] = best_auc
    sweep_history[ctx] = {"val_aucs": val_aucs}
    print(f"Peak AUC for context={ctx}: {best_auc:.4f}")

best_ctx = max(sweep_results, key=sweep_results.get)
print(f"Winner: context={best_ctx} (AUC={sweep_results[best_ctx]:.4f})")
plot_context_sweep(sweep_results, sweep_history, str(SAVE_DIR))

Testing context_size=1


context=1: 100%|██████████| 7110/7110 [00:04<00:00, 1626.89it/s]


  Epoch 1/5 | Val AUC: 0.9754
  Epoch 2/5 | Val AUC: 0.9752
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9775
  Epoch 5/5 | Val AUC: 0.9778
Peak AUC for context=1: 0.9778
Testing context_size=3


context=3: 100%|██████████| 7110/7110 [00:01<00:00, 4246.33it/s]


  Epoch 1/5 | Val AUC: 0.9864
  Epoch 2/5 | Val AUC: 0.9875
  Epoch 3/5 | Val AUC: 0.9880
  Epoch 4/5 | Val AUC: 0.9877
  Epoch 5/5 | Val AUC: 0.9876
Peak AUC for context=3: 0.9880
Testing context_size=5


context=5: 100%|██████████| 7110/7110 [00:01<00:00, 4129.67it/s]


  Epoch 1/5 | Val AUC: 0.9898
  Epoch 2/5 | Val AUC: 0.9902
  Epoch 3/5 | Val AUC: 0.9906
  Epoch 4/5 | Val AUC: 0.9910
  Epoch 5/5 | Val AUC: 0.9914
Peak AUC for context=5: 0.9914
Winner: context=5 (AUC=0.9914)
Context sweep plots saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/sweeps/context
